In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
from Optimizer import Optimizer

In [33]:
import argparse
import datetime
import math
import os
from time import time
from typing import Any, List, Tuple

#import mlflow
import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from loguru import logger
from sparsemax import Sparsemax

from artifacts import (
    build_plot_df_wrapper,
    save_csv_artifact,
    save_plot_strategy,
)
from BayesianOptimizer import BoTorchOptimizer, BoTorchOptimizerVariableStake
from data import (
    apply_final_treatment,
    join_metadata,
    load_metadata_artefacts,
    load_odds,
)
from dependencies.config import load_config
#from dependencies.utils import softmax
from filter import filter_by_linear_combination
from GameProbs import GameProbs

config = load_config("config/config.yml")


def setup(args):
    metadata, gameid_to_outcome = load_metadata_artefacts(config.metadata_path)
    odds = load_odds(config.odds_path, args.bookmakers)
    odds = join_metadata(odds, metadata)

    odds = odds.sort_values(["Datetime", "GameId"], ascending=True)

    odds = odds[
        (odds.Datetime.apply(str) >= args.date_start)
        & (odds.Datetime.apply(str) < args.date_end)
    ]

    return odds, gameid_to_outcome

In [34]:
#args = parser.parse_args()
parser = argparse.ArgumentParser()
parser.add_argument(
    "--date_start",
    type=str,
    default="2019-01-01",
    help="start date of the dataset",
)
parser.add_argument(
    "--date_end",
    type=str,
    default="2024-01-01",
    help="end date of the dataset",
)
parser.add_argument(
    "--bookmakers",
    nargs="+",
    default=None,
    help="A list of strings",
)
parser.add_argument(
    "--aggregator", type=str, help="aggregate by GameId or by Datetime"
)
parser.add_argument(
    "--min_games",
    type=int,
    default=1,
    help="threshold of minimum number of games to enter the optimization task",
)
parser.add_argument(
    "--bets_per_game", type=int, default=5, help="number of bets per game"
)
parser.add_argument(
    "--weight",
    type=float,
    default=0.5,
    help="weight of the linear combination filter",
)
parser.add_argument(
    "--optimizer",
    type=str,
    default="BoTorchOptimizer",
    help="optimizer to use",
)
parser.add_argument(
    "--probability_mapping",
    type=str,
    default="softmax",
    help="probability mapping function to use",
)
parser.add_argument(
    "--n_iterations",
    type=int,
    default=100,
    help="number of iterations to run the optimization task",
)
parser.add_argument(
    "--do_baseline",
    action="store_true",
    help="flag to apply baseline logic or not, not specifying the argument return the opposite of the action",
)
parser.add_argument(
    "--n_jobs",
    type=int,
    default=1,
    help="number of jobs to run in parallel",
)
parser.add_argument(
    "--save_experiment",
    action="store_true",
    help="flag to save the experiment artefacts, not specifying the argument return the opposite of the action",
)
args = parser.parse_args([
"--aggregator", "Datetime",
"--min_games", "1",
"--bets_per_game", "2",
"--n_iterations", "10",
"--weight", "0.5",
"--date_start", "2019-06-01",
"--date_end", "2019-12-01",
#"--do_baseline", "False"
#"--save_experiment"
])
print(args)

odds, gameid_to_outcome = setup(args)

grouped = odds.groupby(args.aggregator)

Namespace(date_start='2019-06-01', date_end='2019-12-01', bookmakers=None, aggregator='Datetime', min_games=1, bets_per_game=2, weight=0.5, optimizer='BoTorchOptimizer', probability_mapping='softmax', n_iterations=10, do_baseline=False, n_jobs=1, save_experiment=False)


In [35]:
for a ,b in grouped:
    print(a)
    print(b)

2019-06-01
        GameId Sportsbook      Market Scenario    Bet     Odd  public_prob  \
56978  2782628     Bet365         h2h            home    1.45     0.689655   
56979  2782628     Bet365         h2h            draw    4.00     0.250000   
56980  2782628     Bet365         h2h            away    7.50     0.133333   
56981  2782628     Bet365  over/under      0.5   over    1.06     0.943396   
56982  2782628     Bet365  over/under      0.5  under   10.00     0.100000   
...        ...        ...         ...      ...    ...     ...          ...   
59081  2782629  NordicBet       exact    5 : 0      o  175.00     0.005714   
59082  2782629  NordicBet       exact    5 : 1      o  200.00     0.005000   
59083  2782629  NordicBet       exact    6 : 0      o  250.00     0.004000   
59084  2782629  NordicBet  both_score             yes    2.26     0.442478   
59085  2782629  NordicBet  both_score              no    1.58     0.632911   

           Home       Away    Datetime  
56978  Flam

In [36]:
# Function to get the n-th group and its DataFrame
def get_nth_group(grouped, n):
    for i, (group_key, group_df) in enumerate(grouped):
        if i == n:
            return group_key, group_df
    raise IndexError("Group index out of range")
n = 0
group = get_nth_group(grouped, n)

In [37]:
date, group_data = group
print(date)

2019-06-01


In [38]:
games_ids = group_data.GameId.unique()
print(games_ids)

['2782628' '2782629']


In [39]:
odds

,GameId,Sportsbook,Market,Scenario,Bet,Odd,public_prob,Home,Away,Datetime
56978,2782628,Bet365,h2h,,home,1.45,0.689655,Flamengo,Fortaleza,2019-06-01
56979,2782628,Bet365,h2h,,draw,4.00,0.250000,Flamengo,Fortaleza,2019-06-01
56980,2782628,Bet365,h2h,,away,7.50,0.133333,Flamengo,Fortaleza,2019-06-01
56981,2782628,Bet365,over/under,0.5,over,1.06,0.943396,Flamengo,Fortaleza,2019-06-01
56982,2782628,Bet365,over/under,0.5,under,10.00,0.100000,Flamengo,Fortaleza,2019-06-01
...,...,...,...,...,...,...,...,...,...,...
366561,3010529,NordicBet,exact,5 : 0,o,100.00,0.010000,Ceará,Athletico-PR,2019-11-30
366562,3010529,NordicBet,exact,5 : 1,o,100.00,0.010000,Ceará,Athletico-PR,2019-11-30
366563,3010529,NordicBet,exact,6 : 0,o,125.00,0.008000,Ceará,Athletico-PR,2019-11-30
366564,3010529,NordicBet,both_score,,yes,1.97,0.507614,Ceará,Athletico-PR,2019-11-30


In [9]:
# metadata, gameid_to_outcome = load_metadata_artefacts("data/metadata-with-date-new.parquet")
# odds = load_odds("data/odds-new-correct.parquet")
# odds = join_metadata(odds, metadata)
# print(metadata.shape)
# print(odds.shape)

In [10]:
# from data import load_map
# data = load_map("data/meanSurface-new.json")

In [40]:
GAME_ID = "7290395" # Chapecoense x Flamengo
my_game = GameProbs(GAME_ID) 
df = my_game.build_dataframe()
df

,0,1,2,3,4,5,6
0,0.0245,0.0006,0.0017,0.0035,0.0058,0.0085,0.1814
1,0.0002,0.0250,0.0029,0.0057,0.0093,0.0130,0.2223
2,0.0002,0.0011,0.0146,0.0054,0.0085,0.0116,0.1732
3,0.0002,0.0009,0.0022,0.0078,0.0061,0.0081,0.1089
4,0.0001,0.0006,0.0014,0.0025,0.0047,0.0049,0.0602
5,0.0001,0.0003,0.0008,0.0014,0.0021,0.0029,0.0306
6,0.0001,0.0004,0.0009,0.0015,0.0021,0.0026,0.0264


In [12]:
group_data

,GameId,Sportsbook,Market,Scenario,Bet,Odd,public_prob,Home,Away,Datetime
56978,2782628,Bet365,h2h,,home,1.45,0.689655,Flamengo,Fortaleza,2019-06-01
56979,2782628,Bet365,h2h,,draw,4.00,0.250000,Flamengo,Fortaleza,2019-06-01
56980,2782628,Bet365,h2h,,away,7.50,0.133333,Flamengo,Fortaleza,2019-06-01
56981,2782628,Bet365,over/under,0.5,over,1.06,0.943396,Flamengo,Fortaleza,2019-06-01
56982,2782628,Bet365,over/under,0.5,under,10.00,0.100000,Flamengo,Fortaleza,2019-06-01
...,...,...,...,...,...,...,...,...,...,...
59081,2782629,NordicBet,exact,5 : 0,o,175.00,0.005714,Bahia,Grêmio,2019-06-01
59082,2782629,NordicBet,exact,5 : 1,o,200.00,0.005000,Bahia,Grêmio,2019-06-01
59083,2782629,NordicBet,exact,6 : 0,o,250.00,0.004000,Bahia,Grêmio,2019-06-01
59084,2782629,NordicBet,both_score,,yes,2.26,0.442478,Bahia,Grêmio,2019-06-01


In [13]:
#GAME_ID = '2738735'
#odds_sample = group_data[(group_data.GameId==GAME_ID)]
odds_sample = group_data
#odds_sample = join_metadata(odds_sample, metadata)
#games_ids = odds_sample['GameId'].unique()
print(games_ids)

# Initialize dict to store dataframes of favorable bet opportunities
odds_dict = {}
# Initialize dict to store 7x7 matrices/dataframes of real probabilities 
df_probs_dict = {}

for game_id in games_ids:
    df = GameProbs(game_id).build_dataframe()
    odds_sample = group_data[(group_data.GameId==game_id)]
    odds_sample = apply_final_treatment(df_odds=odds_sample, df_real_prob=df)
    odds_sample = filter_by_linear_combination(odds_sample, n=5)
    odds_dict[game_id] = odds_sample
    df_probs_dict[game_id] = df
odds_dt = pd.concat(odds_dict.values())

['2782628' '2782629']


In [14]:
len(df_probs_dict.values())

2

In [15]:
odds_sample = odds_sample[odds_sample.Market.isin(['spread', 'over/under', 'h2h', 'exact', 'both_score'])].reset_index(drop=True)
print(odds_sample.shape)

(4, 17)


In [16]:
n = len(odds_dt)
allocation_array = np.round(np.array(((1/n), ) * n), 4)
allocation_array

array([0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111,
       0.1111])

## COBYLA via Simulation Optimizer

In [17]:
from monte_carlo_return import (
    generate_bet_return,
    compute_objective_via_simulation,
)

In [18]:
games_ids

array(['2782628', '2782629'], dtype=object)

In [19]:
len(odds_dict)

2

In [20]:
n = len(odds_dt)
allocation_array = np.round(np.array(((1/n), ) * n), 4)
allocation_array

financial_return_array = generate_bet_return(
    df_prob=df_probs_dict,
    df_bet=odds_dt,
    num_simulations=10000,
    allocation_array=allocation_array
)
financial_return_array                                             

array([0.687709 , 1.870924 , 1.3859725, ..., 0.6460465, 3.249675 ,
       1.3859725])

In [21]:
print(np.mean(financial_return_array))
print(np.std(financial_return_array))

1.1726516120000001
2.3527541642281093


In [22]:
games_ids

array(['2782628', '2782629'], dtype=object)

In [23]:
compute_objective_via_simulation(
    x=allocation_array,
    df_prob=df_probs_dict,
    df_bet=odds_dt,
    num_simulations=10000
)

-0.48073276322967534

In [24]:
solution, value, time_limit_flag = Optimizer().run_optimization(
    fun=compute_objective_via_simulation,
    x0=np.zeros(len(odds_dt)),
    args=(df_probs_dict, odds_dt, 1000),
)
print(f"Value: {value}")

Value: -3.924314369779696
   Normal return from subroutine COBYLA


   NFVALS =  114   F =-3.924314E+00    MAXCV = 0.000000E+00
   X = 1.488675E+00   2.713398E+00   1.750020E+00   1.993636E+00  -1.666307E+00
       8.951997E-01  -5.212408E-01  -1.344915E+00  -2.914300E+00


In [26]:
value

-6.618251607544345

## COBYLA Analytical Optimizer

In [24]:
from analytical_return import compute_objective_via_analytical

In [25]:
odds_favorable = np.array(odds_dt['Odd'])
real_prob_favorable = np.array(odds_dt['real_prob'])
#scenario_favorable = np.array(odds_sample_favorable['Bet'])
event_favorable = list(odds_dt['BetMap'].values)
games_ids = np.array(odds_dt['GameId'])

In [26]:
len(games_ids)

9

In [31]:
# n = len(odds_dt)
# allocation_array = np.round(np.array(((1/n), ) * n), 4)
# allocation_array

In [32]:
allocation_array

array([0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111, 0.1111,
       0.1111])

In [33]:
compute_objective_via_analytical(
    x=np.zeros(len(odds_dt)),
    public_odd=odds_favorable,
    real_probabilities=real_prob_favorable,
    event=event_favorable,
    games_ids=games_ids,
    df_probs_dict=df_probs_dict,
)

-0.48152068573464457

In [34]:
solution, value, time_limit_flag = Optimizer().run_optimization(
    fun=compute_objective_via_analytical,
    x0=np.zeros(len(odds_favorable)),
    args=(odds_favorable, real_prob_favorable, event_favorable, games_ids, df_probs_dict)
)
print(f"Value: {value}")

Value: -6.965906207766195
   Return from subroutine COBYLA because the MAXFUN limit has been reached.


   NFVALS = 1000   F =-6.965906E+00    MAXCV = 0.000000E+00
   X = 4.572379E+00   2.105247E+00   1.509035E+00   2.129818E+00  -2.443923E-01
       3.172832E-01  -1.435522E+00  -2.498112E+00  -3.376482E+00


## LongTermOptimizer

In [27]:
len(odds_favorable) == len(odds_dt)

True

In [28]:
from LongTermOptimizer import estimate_long_term_return
solution, value, time_limit_flag = Optimizer().run_optimization(
    fun=estimate_long_term_return,
    x0=np.zeros(len(odds_dt)),
    args=(df_probs_dict, odds_dt, 100)
)
print(f"Value: {value}")

Value: -1.5048704186575604e+16
   Normal return from subroutine COBYLA


   NFVALS =   87   F =-1.504870E+16    MAXCV = 0.000000E+00
   X = 7.702248E-01   3.335832E+00   1.214765E+00  -5.862824E-01  -2.318410E+00
       1.001839E+00  -5.676557E-01  -7.775075E-02  -6.495089E-01


In [30]:
from dependencies.utils import softmax
assert len(solution) == len(odds_dt)
print(f"Sum of solution: {sum(solution)}")
solution = softmax(solution)
print(f"Sum of solution after softmax: {sum(solution)}")
print(solution)

Sum of solution: 2.123054062013594
Sum of solution after softmax: 1.0
[0.05535704 0.72011537 0.0863444  0.01425769 0.0025223  0.06978494
 0.01452575 0.02370837 0.01338414]
